In [ ]:
import pandas as pd
from joblib import load, dump

from utils.read_answer import read_answer
from utils.compare_with_language import compare_with_language
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

from utils.missing_data import dico_other_iso_to_genus

In [ ]:
name_model = "apertus-lm-7b"

In [ ]:
threshold = 20

# Load maps


In [ ]:
maps_wals2genus: dict[str, str] = load("../multiQ/data/maps/mapping_wals_genus.joblib")
maps_iso2wals: dict[str, str] = load("../multiQ/data/maps/iso_to_wals.joblib")

# Joining tables

In [ ]:
path_switch: str = f"../multiQ/data/{name_model}.csv"
df_switch: pd.DataFrame = pd.read_csv(path_switch)

In [ ]:
#if column "gpt-4_evaluation", rename it eval_completion
if "gpt-4_evaluation" in df_switch.columns:
    df_switch = df_switch.rename(columns={"gpt-4_evaluation": "eval_completion"})

In [ ]:
df_switch

In [ ]:
#path_fidelity: str = f"../multiQ/data/language_fidelity/{name_model}.csv"
path_fidelity : str = f"../multiQ/data/language_fidelity/Llama-2-13b-chat-hf.csv"
df_fidelity: pd.DataFrame = pd.read_csv(path_fidelity)

In [ ]:
df_fidelity

In [ ]:
len(df_fidelity["input_family"].unique())

In [ ]:
#df_fidelity = df_fidelity.reset_index()
#df_switch = df_switch.reset_index()

df_fidelity["id_lg"] = df_fidelity["id"].astype(str).str.cat(df_fidelity["language"].astype(str), sep=";")


df_switch["id_lg"] = df_switch["id"].astype(str).str.cat(df_switch["language"].astype(str), sep=";")

In [ ]:
df_eval = df_fidelity.merge(
    df_switch[[ "id_lg", "eval_completion", "prompt", "prompt_en"]],  # keep only needed columns from df2
    on="id_lg",
    how="left"                        # keeps all rows from df1
)

# Select questions

In [ ]:
prompt_en :  list[str]= df_eval["prompt_en"].unique().tolist()# all questions (in en version)
languages : list[str] = df_eval["iso_639_3"].unique().tolist()

In [ ]:
results = {k: {} for k in prompt_en}

In [ ]:
for i, row in df_eval.iterrows():
    current_en_prompt = row["prompt_en"]
    current_language = row["iso_639_3"]
    try:
        answer = read_answer(row["eval_completion"])
    except Exception as e:
        print(f"Error reading answer for row {i}: {e}")
        results[current_en_prompt][current_language] = None
        continue
    results[current_en_prompt][current_language] = answer

In [ ]:
nb_corr_answ_lg = {lg : 0 for lg in languages}
for question in results:
    for lg_q in results[question]:
        if results[question][lg_q]:
            nb_corr_answ_lg[lg_q] += 1

In [ ]:
list_lg_ab_thresh = [lg for lg in nb_corr_answ_lg if nb_corr_answ_lg[lg] >= threshold]

In [ ]:
#get total nb questions
tot_nb_questions = 0
for lg in list_lg_ab_thresh:
    tot_nb_questions += nb_corr_answ_lg[lg]
print(tot_nb_questions)

In [ ]:
result_language = {}
result_language_prop = {}
for language in list_lg_ab_thresh:
    r_lg, r_lg_prop = compare_with_language(language, results, list_lg_ab_thresh)
    result_language[language] = r_lg
    result_language_prop[language] = r_lg_prop

# Convert to genus

In [ ]:
result_output_genus = {}
nb_questions_input_genus = {}
for lg_input in result_language:
    result_output_genus[lg_input] = {}
    nb_questions_input_genus[lg_input] = {}
    for lg_output in result_language[lg_input]:
        if lg_output in maps_iso2wals:
            wals_lg_output = maps_iso2wals[lg_output]
            if type(wals_lg_output) == str:
                if wals_lg_output in maps_wals2genus:
                    output_genus = maps_wals2genus[wals_lg_output]
                else:
                    print(wals_lg_output)

            else:
                if maps_wals2genus[wals_lg_output[0]] == maps_wals2genus[wals_lg_output[1]]:
                    output_genus = maps_wals2genus[wals_lg_output[0]]
                else:
                    print("list of wals")
                    print(lg_output, wals_lg_output)

        else:
            output_genus = dico_other_iso_to_genus[lg_output]

        if output_genus not in result_output_genus[lg_input]:
            result_output_genus[lg_input][output_genus] = 0
            nb_questions_input_genus[lg_input][output_genus] = 0

        result_output_genus[lg_input][output_genus] += result_language[lg_input][lg_output]
        nb_questions_input_genus[lg_input][output_genus] += nb_corr_answ_lg[lg_input]

In [ ]:
result_output_genus

In [ ]:
nb_questions_input_genus

In [ ]:
result_genus = {}
nb_questions_genus = {}
for lg_input in result_output_genus:
    if lg_input in maps_iso2wals:
        wals_lg_input = maps_iso2wals[lg_input]
        if type(wals_lg_input) == str:
            if wals_lg_input in maps_wals2genus:
                input_genus = maps_wals2genus[wals_lg_input]
            else:
                print(wals_lg_input)

        else:
            if maps_wals2genus[wals_lg_input[0]] == maps_wals2genus[wals_lg_input[1]]:
                input_genus = maps_wals2genus[wals_lg_input[0]]
            else:
                print("list of wals")
                print(lg_input, wals_lg_input)
    else:

        input_genus = dico_other_iso_to_genus[lg_input]


    if input_genus not in result_genus:
        result_genus[input_genus] = result_output_genus[lg_input]
        nb_questions_genus[input_genus] = nb_questions_input_genus[lg_input]
    else:
        for output_genus in result_output_genus[lg_input]:
            if output_genus in result_genus[input_genus]:
                result_genus[input_genus][output_genus] += result_output_genus[lg_input][output_genus]
            else:
                result_genus[input_genus][output_genus] = result_output_genus[lg_input][output_genus]

        for output_genus in nb_questions_input_genus[lg_input]:
            print(output_genus, input_genus)
            if output_genus in nb_questions_genus[input_genus]:
                nb_questions_genus[input_genus][output_genus] += nb_questions_input_genus[lg_input][output_genus]
            else:
                nb_questions_genus[input_genus][output_genus] = nb_questions_input_genus[lg_input][output_genus]

In [ ]:
print(result_genus)
print(nb_questions_genus)

In [ ]:
import copy

In [ ]:
raw_result_genus = copy.deepcopy(result_genus)

In [ ]:
#dividing by number of questions
for input_genus in result_genus:
    for output_genus in result_genus[input_genus]:
        if result_genus[input_genus][output_genus] !=0:
            result_genus[input_genus][output_genus] /= nb_questions_genus[input_genus][output_genus]

# Results

In [ ]:
# Convert the nested dictionary to a DataFrame
df = pd.DataFrame(result_genus).T  # Transpose so rows and columns align correctly

fig, ax = plt.subplots(figsize=(20, 15))
sns.heatmap(df, annot=False, cmap="coolwarm", cbar=True, square=True, vmin=0, vmax=1, ax=ax)

# title/labels
#ax.set_title(f"Threshold: {threshold} avail. questions", fontsize=24, fontweight='bold', pad=20)
ax.set_xlabel("Target Genus", fontsize=25)
ax.set_ylabel("Source Genus", fontsize=25)

# x ticks
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=25)

# y ticks: rotate about the anchor and align so long labels don't slide down
ax.set_yticklabels(ax.get_yticklabels(), rotation=-45,
                   ha="right", va="center", fontsize=25, rotation_mode='anchor')

# give extra left margin so labels don't get clipped
plt.subplots_adjust(left=0.30)

# ensure layout and save
plt.tight_layout()
plt.savefig(f"subfamilies/results/{name_model}/genus_th_{threshold}.png", dpi=300, bbox_inches='tight')

In [ ]:
# computing overall mean, but excluding the diagonal
import numpy as np
mask = np.eye(df.shape[0], dtype=bool)
mean_off_diagonal = df.where(~mask).stack().mean()
print(f"Mean off-diagonal value: {mean_off_diagonal:.4f}")

In [ ]:
# computing diagonal mean
diagonal_mean = df.where(mask).stack().mean()
print(f"Mean diagonal value: {diagonal_mean:.4f}")

In [ ]:
#computing micro average


tot_pairs = 0
sum_scores = 0
for i in result_genus:
    for j in result_genus[i]:
        if i != j:
            tot_pairs+=1
            sum_scores += result_genus[i][j]


print(sum_scores/tot_pairs)
print(sum_scores)
print(tot_pairs)
print(name_model)

In [ ]:
#computing micro average


macro_sum = 0
tot_in_genus = 0

for i in result_genus:
    tot_pairs = 0
    sum_scores = 0
    for j in result_genus[i]:
        if i != j:
            tot_pairs+=1
            sum_scores += result_genus[i][j]
    macro_sum += sum_scores/tot_pairs
    tot_in_genus+=1
print(macro_sum)
print(tot_in_genus)
print(macro_sum/tot_in_genus)
print(name_model)

In [ ]:
for i in result_genus:
    size = sum(1 for j in result_genus[i] if i != j)
    print(f"{i}: {size} pairs")

In [ ]:
results_nb_q = load("results_nb_q.joblib")
results_nb_g = load("results_nb_g.joblib")

In [ ]:
if name_model not in results_nb_q:
    results_nb_q[name_model] = {}
    results_nb_g[name_model] = {}

In [ ]:
results_nb_q[name_model][threshold] = tot_nb_questions
results_nb_g[name_model][threshold] = len(result_genus)

dump(results_nb_q, "results_nb_q.joblib")
dump(results_nb_g, "results_nb_g.joblib")

In [ ]:
results_nb_q

In [ ]:
s = "&"
for model in results_nb_q:
    s += (f"{round(results_nb_q[model][100]/1000)} & {results_nb_g[model][100]} &")

print(s)

In [ ]:
#For each model and threshold print the value results_nb_q rounded to 1000
for model in results_nb_q:
    for th in results_nb_q[model]:
        print(f"{model} - {th} : {round(results_nb_q[model][th]/1000)}k")